# Day 2 · 실습 : ResNet18과 ViT-B/16의 이미지 처리 비교

**연계 이론:** 이론 2 · 4.1~4.4 / 이론 3 · 3.3~3.4, 4.3, 5.1, 6.1~6.3, 7.1 / 이론 4 · 7.4

동일한 원본 이미지를 두 사전학습 모델에 입력하여
**Top-5 예측과 내부 특징 표현**을 비교합니다.

이번 실습에서는 모델을 새로 학습하지 않습니다.

## 1. 사전학습 모델 준비

두 모델 모두 ImageNet-1K 분류 모델입니다.
로컬 `weights/` 폴더에 공식 가중치가 있으면 먼저 사용하고,
없으면 TorchVision의 캐시 또는 다운로드를 사용합니다.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from PIL import Image
from torch.nn import functional as F
from torchvision.models import (
    ResNet18_Weights,
    ViT_B_16_Weights,
    resnet18,
    vit_b_16,
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("실행 장치:", DEVICE)

resnet_weights = ResNet18_Weights.IMAGENET1K_V1
vit_weights = ViT_B_16_Weights.IMAGENET1K_V1


def load_pretrained_model(builder, weight_info):
    filename = weight_info.url.rsplit("/", 1)[-1]

    candidate_paths = [
        Path("weights") / filename,
        Path(filename),
    ]

    local_path = next(
        (p for p in candidate_paths if p.is_file()),
        None
    )

    if local_path is not None:
        model = builder(weights=None)

        state_dict = torch.load(
            local_path,
            map_location="cpu",
            weights_only=True
        )

        model.load_state_dict(state_dict)

        print("로컬 가중치 사용:", local_path)
        return model

    try:
        model = builder(weights=weight_info)

        print(
            "TorchVision 캐시/공식 가중치 사용:",
            filename
        )
        return model

    except Exception as exc:
        raise RuntimeError(
            f"사전학습 가중치를 불러오지 못했습니다: {filename}\n"
            f"공식 .pth 파일을 weights/{filename}에 두거나 "
            "PyTorch 가중치 다운로드 연결을 확인하세요."
        ) from exc


resnet_model = load_pretrained_model(
    resnet18,
    resnet_weights
).to(DEVICE).eval()

vit_model = load_pretrained_model(
    vit_b_16,
    vit_weights
).to(DEVICE).eval()


def count_parameters(model):
    return sum(
        p.numel() for p in model.parameters()
    )


print(
    f"ResNet18 파라미터: "
    f"{count_parameters(resnet_model):,}"
)

print(
    f"ViT-B/16 파라미터: "
    f"{count_parameters(vit_model):,}"
)

## 2. 동일한 원본 이미지와 모델별 전처리

**연계 이론:** 이론 2 · 4.1 / 이론 3 · 2.1, 6.3

같은 원본 사진을 사용하지만, 사전학습 가중치에 맞는
전처리는 모델별로 적용합니다.

`USER_IMAGE_PATH`를 바꾸면 다른 사진으로 다시 비교할 수 있습니다.

In [ ]:
USER_IMAGE_PATH = ""
# 변경해 보기: "assets/my_photo.jpg"

if USER_IMAGE_PATH.strip():
    image_path = Path(USER_IMAGE_PATH)

    if not image_path.is_file():
        raise FileNotFoundError(
            f"이미지 파일을 찾을 수 없습니다: {image_path}"
        )

else:
    candidates = [
        Path("assets/day2_07_sample_dog.jpg"),
        Path("day2_07_sample_dog.jpg"),
    ]

    image_path = next(
        (p for p in candidates if p.is_file()),
        None
    )

    if image_path is None:
        image_path = candidates[0]
        image_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        try:
            urlretrieve(
                "https://raw.githubusercontent.com/"
                "pytorch/hub/master/images/dog.jpg",
                image_path,
            )

        except Exception as exc:
            raise RuntimeError(
                "예제 이미지 다운로드에 실패했습니다. "
                "사진을 assets/day2_07_sample_dog.jpg에 저장하거나 "
                "USER_IMAGE_PATH에 사진 경로를 입력하세요."
            ) from exc


with Image.open(image_path) as source:
    image = source.convert("RGB")

print(
    "원본 이미지:",
    image_path,
    "|",
    image.size
)

plt.figure(figsize=(5.5, 4.5))
plt.imshow(image)
plt.title("Original image (same source for both models)")
plt.axis("off")
plt.show()


# 모델별 권장 전처리
resnet_preprocess = resnet_weights.transforms()
vit_preprocess = vit_weights.transforms()

resnet_input = (
    resnet_preprocess(image)
    .unsqueeze(0)
    .to(DEVICE)
)

vit_input = (
    vit_preprocess(image)
    .unsqueeze(0)
    .to(DEVICE)
)

print(
    "ResNet18 입력 Shape:",
    tuple(resnet_input.shape)
)

print(
    "ViT-B/16 입력 Shape:",
    tuple(vit_input.shape)
)

assert resnet_input.shape == (1, 3, 224, 224)
assert vit_input.shape == (1, 3, 224, 224)

## 3. 모델 추론과 중간 출력 추출

**연계 이론:** 이론 2 · 4.2~4.4 / 이론 3 · 3.4, 5.1, 6.1~6.3

ResNet은 마지막 `layer4`만,
ViT는 `conv_proj`와 `encoder`의 출력을 저장합니다.

실습 ③에서 수행했던 `layer1~4` 전체 관찰은 반복하지 않습니다.

In [ ]:
features = {}


def capture_output(name):
    def hook(module, inputs, output):
        features[name] = output.detach().cpu()

    return hook


handles = [
    resnet_model.layer4.register_forward_hook(
        capture_output("resnet_layer4")
    ),

    vit_model.conv_proj.register_forward_hook(
        capture_output("vit_patch_embedding")
    ),

    vit_model.encoder.register_forward_hook(
        capture_output("vit_encoder")
    ),
]


try:
    with torch.inference_mode():
        resnet_logits = resnet_model(resnet_input)
        vit_logits = vit_model(vit_input)

finally:
    for handle in handles:
        handle.remove()


# 최종 분류 출력 검증
assert resnet_logits.shape == (1, 1000)
assert vit_logits.shape == (1, 1000)

# 중간 특징 출력 검증
assert features["resnet_layer4"].shape == (
    1, 512, 7, 7
)

assert features["vit_patch_embedding"].shape == (
    1, 768, 14, 14
)

assert features["vit_encoder"].shape == (
    1, 197, 768
)


print(
    "ResNet 분류 출력:",
    tuple(resnet_logits.shape)
)

print(
    "ViT 분류 출력:",
    tuple(vit_logits.shape)
)

print("중간 출력 저장 완료")

## 4. 이미지 분류 결과 비교 — Top-5

같은 사진이라도 모델 구조·학습된 가중치·전처리에 따라
예측 점수가 다를 수 있습니다.

**한 장의 사진만으로 모델의 전체 성능 우열을 판단할 수는 없습니다.**

In [ ]:
def get_top5(logits, weight_info):
    probabilities = torch.softmax(
        logits[0].float(),
        dim=0
    ).detach().cpu()

    top_probabilities, top_indices = torch.topk(
        probabilities,
        k=5
    )

    labels = weight_info.meta["categories"]

    return [
        (
            labels[index.item()],
            probability.item()
        )
        for index, probability in zip(
            top_indices,
            top_probabilities
        )
    ]


resnet_top5 = get_top5(
    resnet_logits,
    resnet_weights
)

vit_top5 = get_top5(
    vit_logits,
    vit_weights
)


# 모델별 Top-5 출력
for model_name, results in [
    ("ResNet18", resnet_top5),
    ("ViT-B/16", vit_top5)
]:
    print(f"\n[{model_name} Top-5]")

    for rank, (label, probability) in enumerate(
        results,
        start=1
    ):
        print(
            f"{rank}. {label}: {probability:.2%}"
        )


# 두 모델의 Top-5 시각화
fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 4.5)
)

for ax, results, model_name in zip(
    axes,
    [resnet_top5, vit_top5],
    ["ResNet18", "ViT-B/16"]
):
    names = [
        label for label, probability in results
    ][::-1]

    values = [
        probability for label, probability in results
    ][::-1]

    ax.barh(names, values)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Softmax probability")
    ax.set_title(f"{model_name} — Top-5")

plt.tight_layout()
plt.show()


print(
    "\nTop-1 일치 여부:",
    resnet_top5[0][0] == vit_top5[0][0]
)

## 5. 실제 내부 특징 표현 비교

**연계 이론:** 이론 2 · 4.2, 4.4 / 이론 3 · 3.4, 4.3, 6.1~6.3

ResNet의 마지막 Feature Map은 `(N, C, H, W)` 형태이고,
ViT Encoder 출력은 `(N, Token 수, Embedding 차원)` 형태입니다.

In [ ]:
resnet_feature = features["resnet_layer4"]
vit_patch_feature = features["vit_patch_embedding"]
vit_tokens = features["vit_encoder"]

# CLS Token과 이미지 Patch Token 분리
cls_token = vit_tokens[:, 0, :]
patch_tokens = vit_tokens[:, 1:, :]


print(
    "ResNet layer4:       ",
    tuple(resnet_feature.shape)
)

print(
    "ViT Patch Embedding: ",
    tuple(vit_patch_feature.shape)
)

print(
    "ViT Encoder:         ",
    tuple(vit_tokens.shape)
)

print(
    "ViT CLS Token:       ",
    tuple(cls_token.shape)
)

print(
    "ViT Patch Tokens:    ",
    tuple(patch_tokens.shape)
)


assert cls_token.shape == (1, 768)
assert patch_tokens.shape == (1, 196, 768)

## 6. 특징 표현 시각화

**연계 이론:** 이론 2 · 4.4 / 이론 3 · 3.3~3.4, 6.1 / 이론 4 · 7.4

ResNet의 채널 평균 Feature Map,
ViT Patch Embedding의 크기,
ViT CLS–Patch Cosine Similarity를 비교합니다.

세 그림은 서로 다른 값을 나타냅니다.
특히 **Cosine Similarity는 Attention Weight가 아닙니다.**

In [ ]:
# 1. ResNet layer4
# 채널 방향 평균으로 7x7 Feature Map 시각화
resnet_map = (
    resnet_feature[0]
    .mean(dim=0)
    .numpy()
)


# 2. ViT Patch Embedding
# 채널별 RMS 크기로 14x14 특징 시각화
vit_patch_magnitude = (
    vit_patch_feature[0]
    .square()
    .mean(dim=0)
    .sqrt()
    .numpy()
)


# 3. ViT Encoder
# CLS Token과 각 Patch Token의 코사인 유사도
cls = cls_token[0]
patches = patch_tokens[0]

similarity_map = F.cosine_similarity(
    patches,
    cls.unsqueeze(0),
    dim=1
).reshape(14, 14).numpy()


# ViT에 실제로 입력된 이미지 복원
# 시각화 목적의 정규화 해제
mean = torch.tensor(
    vit_preprocess.mean
).view(3, 1, 1)

std = torch.tensor(
    vit_preprocess.std
).view(3, 1, 1)

vit_display = (
    vit_input[0].detach().cpu() * std + mean
).clamp(0, 1).permute(1, 2, 0).numpy()


# 네 가지 결과를 함께 표시
fig, axes = plt.subplots(
    1, 4,
    figsize=(16, 4.2)
)

axes[0].imshow(vit_display)
axes[0].set_title("ViT input image")
axes[0].axis("off")


visualizations = [
    (
        axes[1],
        resnet_map,
        "ResNet layer4 — channel mean",
        "viridis"
    ),
    (
        axes[2],
        vit_patch_magnitude,
        "ViT — patch embedding magnitude",
        "viridis"
    ),
    (
        axes[3],
        similarity_map,
        "ViT — CLS/patch cosine similarity",
        "coolwarm"
    ),
]


for ax, data, label, cmap in visualizations:
    im = ax.imshow(
        data,
        cmap=cmap
    )

    ax.set_title(
        label,
        fontsize=10
    )

    ax.axis("off")

    fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )

plt.tight_layout()
plt.show()


print(
    "ResNet Feature Map의 크기:",
    tuple(resnet_map.shape)
)

print(
    "ViT Patch Embedding Map의 크기:",
    tuple(vit_patch_magnitude.shape)
)

print(
    "ViT CLS–Patch 유사도 Map의 크기:",
    tuple(similarity_map.shape)
)

## 7. 종합 비교와 결과 해석

Top-1 결과와 내부 Tensor Shape을 근거로
두 모델의 특징 표현 방식을 설명해 보세요.

시각화된 맵을 곧바로 예측 근거나 중요도 지도로
해석해서는 안 됩니다.

In [ ]:
print("=" * 60)
print("ResNet18 vs ViT-B/16 — 종합 비교")
print("=" * 60)

print("\n[동일 원본 사진의 Top-1]")

print(
    f"ResNet18: "
    f"{resnet_top5[0][0]} "
    f"({resnet_top5[0][1]:.2%})"
)

print(
    f"ViT-B/16: "
    f"{vit_top5[0][0]} "
    f"({vit_top5[0][1]:.2%})"
)


print("\n[내부 특징 표현]")

print(
    "ResNet18 layer4: ",
    tuple(resnet_feature.shape),
    "— Feature Map"
)

print(
    "ViT-B/16 encoder:",
    tuple(vit_tokens.shape),
    "— Token Sequence"
)


print("\n[모델 크기: 파라미터 수]")

print(
    f"ResNet18: "
    f"{count_parameters(resnet_model):,}"
)

print(
    f"ViT-B/16: "
    f"{count_parameters(vit_model):,}"
)


print(
    "\n다른 사진으로 비교하려면 "
    "2장의 USER_IMAGE_PATH를 바꾸고 "
    "이후 셀을 재실행하세요."
)

## 8. 이번 실습 간단하게 요약

**첫째,** 두 모델 모두 같은 원본 이미지에 대해
1000개 클래스 점수를 출력하지만,
예측 클래스와 Softmax 확률은 달라질 수 있습니다.

**둘째,** ResNet은 `(N, C, H, W)` 형태의 Feature Map을 사용하고,
ViT는 이미지 Patch를 Token으로 처리하여
`(N, Token 수, D)` 형태로 표현합니다.

**셋째,** 시각화 결과는 내부 특징을 관찰하는 자료입니다.
CLS–Patch Cosine Similarity는 Self-Attention Weight와 다르며,
단일 이미지의 예측 결과만으로 두 모델의 전체 성능을 비교할 수 없습니다.